AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 04. 증강 CNN 학습하고 성능 비교하기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- 기존 CNN과 같은 조건으로 회전 증강 CNN을 학습할 수 있다.
- 저장된 기존 모델과 증강 모델을 동일한 테스트 데이터로 평가할 수 있다.
- 정확도 차이를 관찰하고, 데이터 증강의 효과를 성급하게 단정하지 않는다.

### VS Code 실행 준비

`DATA_ROOT`에 4강 `Train`과 `Test`가 필요합니다. `BASELINE_MODEL`에는 4강에서 저장한 모델 경로를 지정하세요. 이번 모델은 `models/mission04`에 저장합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers

COURSE_DIR = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_05강')
DATA_ROOT = COURSE_DIR.parent / "wafermap_image_data_261021"

IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]
BATCH_SIZE = 32
EPOCHS = 10
TRAIN_DIR = DATA_ROOT / "Train"
TEST_DIR = DATA_ROOT / "Test"
BASELINE_MODEL = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_04강\\wm_cnn_model_261019.keras')

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def check_image_folders(directory):
    missing = [
        name for name in class_names
        if not any(
            path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            for path in (directory / name).rglob("*")
        )
    ]
    if missing:
        raise FileNotFoundError(
            f"{directory}에 이미지가 없는 클래스: {', '.join(missing)}. "
            "첫 셀의 DATA_ROOT를 4강 Train/Test가 있는 폴더로 설정하세요."
        )

In [ ]:
import numpy as np
import tensorflow as tf

def list_images(directory):
    check_image_folders(directory)
    image_paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_paths = sorted(
            path for path in (directory / class_name).rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        image_paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return image_paths, np.asarray(labels, dtype=np.int32)

def decode_image(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

def make_dataset(image_paths, labels, shuffle=False):
    # Python이 파일을 읽어 Windows의 한글 경로를 처리합니다.
    image_bytes = [path.read_bytes() for path in image_paths]
    dataset = tf.data.Dataset.from_tensor_slices((image_bytes, labels))
    if shuffle:
        dataset = dataset.shuffle(BATCH_SIZE * 8, seed=42)
    dataset = dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_paths, train_labels = list_images(TRAIN_DIR)
test_paths, test_labels = list_images(TEST_DIR)

# 미션 02와 같은 시드 42와 80:20 비율로 분할합니다.
indices = np.random.RandomState(42).permutation(len(train_paths))
val_count = int(len(indices) * 0.2)
train_indices, val_indices = indices[:-val_count], indices[-val_count:]

train_ds = make_dataset([train_paths[i] for i in train_indices], train_labels[train_indices], shuffle=True)
val_ds = make_dataset([train_paths[i] for i in val_indices], train_labels[val_indices])
test_ds = make_dataset(test_paths, test_labels)

print(f"학습 이미지: {len(train_indices):,}장 / 검증 이미지: {len(val_indices):,}장")
print(f"테스트 이미지: {len(test_paths):,}장")

In [ ]:
# 회전 증강 레이어 생성
data_augmentation = keras.Sequential([
    layers.RandomRotation(
        factor=0.25,
        fill_mode="constant",
        fill_value=0.0,
        interpolation="nearest"
    )
])

In [ ]:
# Mission 01에서 data_augmentation을 이미 생성했다고 가정
model_aug = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    data_augmentation,  # 학습 시에만 회전 변형 적용           # 추가: 회전 증강
    layers.Rescaling(1./255),      # 기존: 정규화

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(class_names), activation="softmax"),
])

## 1. 이번 실험에서 확인할 질문

Mission 03에서 4강 CNN 앞에 회전 증강 레이어를 추가해 `model_aug`를 만들었다. 이번에는 새 모델을 학습하여 **회전 증강이 웨이퍼맵 분류 성능에 어떤 영향을 주는지** 확인한다.

4강 CNN 회전 증강 없음 · 기존 학습 모델

↕ 비교

5강 CNN 회전 범위 ±90° · 새로 학습할 모델

↓

동일한 test_ds로 평가 테스트 정확도 · 손실 비교

실험 가설의 예: “회전 증강을 적용하면 원본에 없는 방향의 결함 패턴까지 학습하여 정확도가 높아질 것이다.” 반대로 성능이 낮아질 가능성도 있다. **어떤 결과가 나와도 관찰하고 해석하는 것이 실험의 목적**이다.

## 2. 학습 조건 확인하기

한 실험에서 바꾸려는 조건은 **회전 증강 적용 여부**이다. 기존 모델과 비교할 때 CNN 구조와 학습 설정을 같게 유지해야 차이를 해석하기 쉽다.

| 항목 | 4강 기존 CNN | 5강 증강 CNN |
| --- | --- | --- |
| 입력 이미지 | 64 × 64 × 3 | 동일 |
| 학습 데이터 | `train_ds` | 동일 |
| 검증 데이터 | `val_ds` | 동일 |
| 테스트 데이터 | `test_ds` | 동일 |
| 배치 크기 / 에포크 | 32 / 10 | 동일 |
| CNN 구조 / 학습 설정 | 4강 설정 | 동일 |
| **회전 증강** | **없음** | **±90°** |

**시작 전 확인**

1. `model_aug.summary()`에서 증강 레이어와 기존 CNN 구조를 확인한다.
2. `train_ds`, `val_ds`, `test_ds`가 생성되어 있는지 확인한다.
3. 4강 모델 파일의 저장 위치를 확인한다. 비교할 때 다시 불러올 예정이다.

이번 미션에서는 먼저 한 번 학습하고 결과를 관찰한다. **가중치 초기화, 셔플 등의 무작위성을 통제하는 시드 고정 실험은 이후 미션**에서 다룬다. 따라서 여기서 생긴 정확도 차이를 오직 회전 증강 때문이라고 확정할 수는 없다.

## 3. 증강 CNN 학습하기
4강 모델에서 사용한 손실 함수, 옵티마이저, 평가 지표를 확인하고 새 모델에도 동일하게 설정한다. 다음 코드를 작성해보자.

In [ ]:
# 기존 CNN과 동일한 학습 설정
model_aug.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

`compile()`은 학습 방법을 설정하고, `fit()`은 실제 학습을 진행한다. 학습에 사용하는 것은 원본 `train_ds`지만, 모델 내부의 `RandomRotation`이 훈련 단계에서 이미지를 무작위로 회전시킨다.

In [ ]:
# 학습 과정의 에포크별 결과를 history_aug에 기록
history_aug = model_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

`history_aug`에는 에포크마다 계산된 학습·검증 정확도와 손실이 기록된다. 학습이 끝나면 다음 코드로 확인해보자.

In [ ]:
print(history_aug.history.keys())

`model_aug`는 **새로 생성한 모델**이어야 한다. 4강의 학습된 모델을 계속 학습시키면 “회전 증강을 추가한 새 모델”과의 비교가 아니라 추가 학습 효과까지 포함된 실험이 된다.

## 4. 기존 모델과 증강 모델 평가하기
4강 모델이 현재 메모리에 없다면 저장한 `.keras` 파일을 불러오자. 아래 경로를 실제 저장 경로에 맞게 수정한다.

In [ ]:
try:
    model = keras.models.load_model(BASELINE_MODEL, compile=False)
except (TypeError, ValueError) as error:
    if "quantization_config" not in str(error):
        raise

    # 저장 당시와 같은 CNN 구조에 4강 모델의 가중치를 불러옵니다.
    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Rescaling(1./255),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(len(class_names), activation="softmax"),
    ])
    model.load_weights(BASELINE_MODEL)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

두 모델 모두 **동일한 `test_ds`**로 평가한다. `evaluate()`에서는 증강 레이어가 추론 모드로 실행되므로 테스트 이미지는 회전하지 않는다.

In [ ]:
# 기존 CNN
loss, acc = model.evaluate(test_ds, verbose=0)

# 회전 증강 CNN
loss_aug, acc_aug = model_aug.evaluate(test_ds, verbose=0)

print(f"기존 CNN 정확도: {acc:.4f}")
print(f"증강 CNN 정확도: {acc_aug:.4f}")

**실험 결과 기록**

1. 기존 CNN의 정확도를 기록한다.
2. 증강 CNN의 정확도를 기록한다.
3. 두 결과 중 어느 쪽이 높았는지 확인한다. 결과를 예상과 비교한다.

| 모델 | 회전 증강 | 테스트 정확도 (직접 기록) |
| --- | --- | --- |
| 기존 CNN | 없음 | __________ |
| 증강 CNN | ±90° | __________ |

## 5. 학습 모델 저장하기
학습에 시간이 걸린 모델은 파일로 저장해두면 필요할 때 다시 불러와 평가할 수 있다. 회전 조건을 알아볼 수 있는 파일명을 사용하자.

In [ ]:
MISSION04_MODEL_DIR = COURSE_DIR / "models" / "mission04"
MISSION04_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_file = MISSION04_MODEL_DIR / "cnn_rotation_90.keras"
model_aug.save(model_file)
print("증강 CNN 저장:", model_file)

Colab에서 작업한다면 Google Drive에 저장되도록 경로를 지정해야 런타임 초기화 후에도 파일을 유지할 수 있다. `.keras` 파일에는 모델 구조와 가중치 등이 저장되지만, `history_aug.history`에 담긴 전체 에포크별 기록은 자동으로 보관되지 않는다.

**선택:** 학습 기록까지 남기려면 `pd.DataFrame(history_aug.history).to_csv("cnn_rotation_90_history.csv", index=False)`처럼 별도의 CSV 파일로 저장할 수 있다.

## 6. 실행 결과 해석하기

전체 정확도는 테스트 이미지 중 정답을 맞힌 비율이다. 하지만 정확도 하나만으로 **어느 불량 유형을 더 잘 구분하게 되었는지** 알 수는 없다.

**전체 정확도**

모델의 전체적인 분류 성공 비율을 나타낸다.

**클래스별 결과**

Loc

Scratch

Edge-Ring

예시 그림: 클래스에 따라 성능이 다를 수 있다. 실제 측정값이 아니다.

이번 미션에서는 정확도 변화를 먼저 관찰한다. **증강으로 성능이 향상될 것이라는 가설과 다르더라도 잘못된 실험이 아니다.** 결과를 토대로 어떤 추가 분석이 필요한지 생각해보자.

## 7. 코드 이해 및 결과 분석 활동

**Q4.1.**

`model_aug.compile()`에서 옵티마이저, 손실 함수, 평가 지표의 설정값을 찾으시오. 4강 모델과 동일하게 지정한 이유는 무엇인가?

**답변**

옵티마이저는 **Adam**, 손실 함수는 **`sparse_categorical_crossentropy`**, 평가 지표는 **`accuracy`**입니다. 학습 조건을 맞춰 회전 증강에 따른 차이를 비교합니다.

**Q4.2.**

`model_aug.fit()`에 전달한 훈련 데이터셋과 검증 데이터셋의 변수명을 각각 찾으시오. 모델에 증강 레이어가 있어도 검증 이미지에는 회전이 적용되지 않는 이유는 무엇인가?

**답변**

훈련에는 `train_ds`, 검증에는 `val_ds`를 전달합니다. 검증은 추론 모드로 동작하므로 `RandomRotation`이 이미지를 회전하지 않습니다.

**Q4.3.**

기존 CNN과 증강 CNN의 테스트 정확도를 비교하여 차이를 작성하시오. 데이터 증강에 관한 예상과 실제 결과는 일치했는가?

**답변**

**검증 예시(테스트 15장, 증강 모델 1에포크)**에서 기존 CNN은 **66.67%**, 증강 CNN은 **20.00%**였습니다. 증강 모델이 약 **46.67%p 낮아** 정확도가 높아질 것이라는 예상과 달랐습니다. 수업 데이터로 실행한 뒤 출력된 `acc`와 `acc_aug`를 비교하세요.

**Q4.4.**

두 모델의 전체 정확도가 비슷하다고 가정하자. 이 결과만으로 모든 불량 유형의 분류 성능도 비슷하다고 말할 수 있을까? 추가로 확인하고 싶은 정보를 작성하시오.

**답변**

전체 정확도가 비슷해도 유형별 성능은 다를 수 있습니다. **클래스별 Precision·Recall과 혼동행렬**을 확인해 어떤 유형을 놓치거나 혼동했는지 비교합니다.

**Q4.5.**

학습된 모델 파일을 저장하는 코드와 불러오는 코드를 각각 찾으시오. Colab 런타임이 종료된 후에도 모델을 사용하려면 저장 위치를 어떻게 정해야 하는가?

**답변**

`model_aug.save(model_file)`로 저장하고 `keras.models.load_model(BASELINE_MODEL, compile=False)`로 불러옵니다. Colab에서는 런타임 종료 뒤에도 파일이 남도록 연결한 Google Drive 경로에 저장합니다.